# MLflow + DagsHub — Regression Lab

This notebook mirrors the classification pipeline (`MLFlow_dagshub.ipynb`) but for a **regression** task.

**Pipeline:**
1. Connect MLflow to a remote **DagsHub** tracking server
2. Load a regression dataset (sklearn Diabetes — built-in, no download needed)
3. Train several regression models
4. Log every experiment (params + metrics + model artifact) to DagsHub
5. Register the best model, then promote it to **Production**

Every run and registered model will be visible online in your DagsHub repo under the **Experiments** and **Models** tabs.

## 1. Importing Packages

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from xgboost import XGBRegressor

import mlflow
import mlflow.sklearn
import mlflow.xgboost

import dagshub
import os

## 2. DagsHub MLflow Setup

**One-time setup before running this cell:**

1. Create a free account at https://dagshub.com
2. Create a new repository (e.g. `MLflow-Regression`)
3. Open the repo → click the green **Remote** button → **Experiments** tab → it shows your MLflow tracking URL and credentials

Then replace `repo_owner` and `repo_name` below with **your own** username and repo name.

`dagshub.init(..., mlflow=True)` automatically points MLflow at your DagsHub tracking server and handles authentication (it will prompt you to log in / paste a token the first time).

In [2]:
# TODO: replace with YOUR DagsHub username and repo name
dagshub.init(
    repo_owner="Nithilan77",
    repo_name="DevOps",
    mlflow=True
)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=d63c8eb7-5e64-40b8-a4fe-b97044f66c22&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=42573c7a8df206d99e846b00313cc16fac3e5d70155d309eb7ec09468e92c70f




Accessing as Nithilan77

Initialized MLflow to track repo "Nithilan77/DevOps"

Repository Nithilan77/DevOps initialized!

### (Alternative) Token-based auth

If `dagshub.init` doesn't prompt correctly in your environment, you can set the tracking URI and credentials manually instead. Get the token from DagsHub → your repo → **Remote → Experiments**, or from https://dagshub.com/user/settings/tokens

Uncomment and fill in if needed:

In [3]:
# os.environ["MLFLOW_TRACKING_URI"] = "https://dagshub.com/YOUR_USERNAME/YOUR_REPO.mlflow"
# os.environ["MLFLOW_TRACKING_USERNAME"] = "YOUR_USERNAME"
# os.environ["MLFLOW_TRACKING_PASSWORD"] = "YOUR_DAGSHUB_TOKEN"
# mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])

In [4]:
mlflow.set_experiment("Diabetes Regression PBLM 1")

2026/08/06 20:14:16 INFO mlflow.tracking.fluent: Experiment with name 'Diabetes Regression PBLM 1' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/967034fb37a541979a904c3a40e52e8e', creation_time=1786027457857, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1786027457857, lifecycle_stage='active', name='Diabetes Regression PBLM 1', tags={}, trace_location=None, workspace='default'>

## 3. Data Loading and Processing

Using the sklearn **Diabetes** dataset — a standard regression benchmark. The target is a continuous disease-progression score (not a class label), which is what makes this a regression problem instead of classification.

In [5]:
data = load_diabetes()
X = data.data
y = data.target

print("Shape:", X.shape)
print("Number of features:", X.shape[1])
print("Target range: min =", round(y.min(), 2), "| max =", round(y.max(), 2))
print("Target mean:", round(y.mean(), 2))

Shape: (442, 10)
Number of features: 10
Target range: min = 25.0 | max = 346.0
Target mean: 152.13


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42
)

print("Train size:", X_train.shape[0])
print("Test size :", X_test.shape[0])

Train size: 309
Test size : 133


## 4. Build Models

Note the difference from classification: there is no SMOTE / class balancing here (regression targets are continuous, so there are no classes to balance). Instead we compare a linear baseline against tree-based ensembles.

In [7]:
models = [
    (
        "Linear Regression",
        LinearRegression()
    ),
    (
        "Random Forest",
        RandomForestRegressor(
            n_estimators=100,
            max_depth=5,
            random_state=42
        )
    ),
    (
        "Gradient Boosting",
        GradientBoostingRegressor(
            n_estimators=200,
            max_depth=3,
            learning_rate=0.05,
            random_state=42
        )
    ),
    (
        "XGBoost",
        XGBRegressor(
            n_estimators=200,
            max_depth=3,
            learning_rate=0.05,
            random_state=42
        )
    )
]

### Regression metrics helper

For regression we track:
- **RMSE** (Root Mean Squared Error) — lower is better
- **MAE** (Mean Absolute Error) — lower is better
- **R2** (coefficient of determination) — higher is better, best = 1.0

These replace accuracy / precision / recall / F1 from the classification notebook.

In [8]:
def evaluate_regression(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return {"RMSE": rmse, "MAE": mae, "R2": r2}

In [9]:
reports = []
trained_models = []

for model_name, model in models:
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    report = evaluate_regression(y_test, predictions)
    reports.append(report)
    trained_models.append(model)

    print("=" * 50)
    print(model_name)
    print("=" * 50)
    print("RMSE:", round(report["RMSE"], 4))
    print("MAE :", round(report["MAE"], 4))
    print("R2  :", round(report["R2"], 4))
    print()

Linear Regression
RMSE: 53.1202
MAE : 41.9194
R2  : 0.4773

Random Forest
RMSE: 52.6204
MAE : 42.1809
R2  : 0.4871

Gradient Boosting
RMSE: 54.8022
MAE : 44.0128
R2  : 0.4437

XGBoost
RMSE: 54.1547
MAE : 43.0233
R2  : 0.4567



## 5. Log All Experiments to DagsHub

Each model becomes one MLflow **run** logged to your DagsHub repo, capturing its hyperparameters, the three regression metrics, and the serialized model artifact. After this cell runs, open your repo's **Experiments** tab on DagsHub to see them side by side.

In [10]:
for i, (model_name, _) in enumerate(models):
    report = reports[i]
    model = trained_models[i]

    with mlflow.start_run(run_name=model_name):

        # Params
        mlflow.log_param("Model", model_name)
        mlflow.log_params(model.get_params())

        # Metrics
        mlflow.log_metric("RMSE", report["RMSE"])
        mlflow.log_metric("MAE", report["MAE"])
        mlflow.log_metric("R2", report["R2"])

        # Tags
        mlflow.set_tag("Task", "Regression")
        mlflow.set_tag("Dataset", "sklearn Diabetes")

        # Model artifact
        if "XGBoost" in model_name:
            mlflow.xgboost.log_model(model, artifact_path="model")
        else:
            mlflow.sklearn.log_model(model, artifact_path="model")

print("All experiments logged to DagsHub!")

2026/08/06 20:14:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Linear Regression at: https://dagshub.com/Nithilan77/DevOps.mlflow/#/experiments/0/runs/68f91610a2464aa089674040ce76a265
🧪 View experiment at: https://dagshub.com/Nithilan77/DevOps.mlflow/#/experiments/0


2026/08/06 20:15:00 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Random Forest at: https://dagshub.com/Nithilan77/DevOps.mlflow/#/experiments/0/runs/bb4c454745a64fd2bfe047150d420719
🧪 View experiment at: https://dagshub.com/Nithilan77/DevOps.mlflow/#/experiments/0


2026/08/06 20:15:28 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Gradient Boosting at: https://dagshub.com/Nithilan77/DevOps.mlflow/#/experiments/0/runs/5576818881f243c0a7292a9c5c6d2ce0
🧪 View experiment at: https://dagshub.com/Nithilan77/DevOps.mlflow/#/experiments/0


2026/08/06 20:16:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBoost at: https://dagshub.com/Nithilan77/DevOps.mlflow/#/experiments/0/runs/faf243661ef844e8adffc4921e6c5cd1
🧪 View experiment at: https://dagshub.com/Nithilan77/DevOps.mlflow/#/experiments/0
All experiments logged to DagsHub!


## 6. Select and Register the Best Model

We pick the model with the **highest R2** as the champion (for RMSE/MAE you would instead pick the minimum). Registering it creates a versioned entry under the **Models** tab in DagsHub.

In [11]:
best_index = int(np.argmax([r["R2"] for r in reports]))

best_model_name = models[best_index][0]
best_model = trained_models[best_index]
best_report = reports[best_index]

print("Best Model:", best_model_name)
print("RMSE:", round(best_report["RMSE"], 4))
print("MAE :", round(best_report["MAE"], 4))
print("R2  :", round(best_report["R2"], 4))

Best Model: Random Forest
RMSE: 52.6204
MAE : 42.1809
R2  : 0.4871


In [12]:
with mlflow.start_run(run_name=f"Champion_{best_model_name}") as run:

    # Params
    mlflow.log_param("Model", best_model_name)
    mlflow.log_param("Selection_Metric", "R2 Score")
    mlflow.log_param("Dataset", "sklearn Diabetes")
    mlflow.log_params(best_model.get_params())

    # Metrics
    mlflow.log_metric("RMSE", best_report["RMSE"])
    mlflow.log_metric("MAE", best_report["MAE"])
    mlflow.log_metric("R2", best_report["R2"])

    # Tags
    mlflow.set_tag("Model_Type", best_model_name)
    mlflow.set_tag("Stage", "Candidate")
    mlflow.set_tag("Task", "Regression")

    # Register the model
    if "XGBoost" in best_model_name:
        model_info = mlflow.xgboost.log_model(
            best_model,
            artifact_path="model",
            registered_model_name="Diabetes_Best_Model"
        )
    else:
        model_info = mlflow.sklearn.log_model(
            best_model,
            artifact_path="model",
            registered_model_name="Diabetes_Best_Model"
        )

    run_id = run.info.run_id
    model_uri = model_info.model_uri

print("Run ID    :", run_id)
print("Model URI :", model_uri)
print("Model Name:", "Diabetes_Best_Model")

2026/08/06 20:17:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Successfully registered model 'Diabetes_Best_Model'.
2026/08/06 20:17:40 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Diabetes_Best_Model, version 1
Created version '1' of model 'Diabetes_Best_Model'.


🏃 View run Champion_Random Forest at: https://dagshub.com/Nithilan77/DevOps.mlflow/#/experiments/0/runs/fe3bd040d94d46d290288e4cdfb6e3c6
🧪 View experiment at: https://dagshub.com/Nithilan77/DevOps.mlflow/#/experiments/0
Run ID    : fe3bd040d94d46d290288e4cdfb6e3c6
Model URI : models:/m-1b785d4a2c1c4828a132952748bfebdc
Model Name: Diabetes_Best_Model


## 7. Load the Registered Model and Verify

Load the model back from the registry (by name + version) and confirm it predicts correctly — this proves the artifact stored on DagsHub is usable.

In [13]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

model_name = "Diabetes_Best_Model"
latest_version = client.get_latest_versions(model_name)[0]

print("Model Name:", latest_version.name)
print("Version   :", latest_version.version)
print("Stage     :", latest_version.current_stage)

Model Name: 120557_Diabetes_Best_Model
Version   : 1
Stage     : None


In [14]:
model_uri = f"models:/{model_name}/{latest_version.version}"

loaded_model = mlflow.pyfunc.load_model(model_uri)
print("Model loaded successfully")

predictions = loaded_model.predict(X_test)
print("First 10 predictions:", np.round(predictions[:10], 2))

Model loaded successfully
First 10 predictions: [160.3  176.97 142.26 248.5  114.01 119.71 249.82 212.23 134.01 171.91]


In [15]:
from sklearn.metrics import mean_squared_error, r2_score

rmse = np.sqrt(mean_squared_error(y_test, predictions))
print("Reloaded model RMSE:", round(rmse, 4))
print("Reloaded model R2  :", round(r2_score(y_test, predictions), 4))

Reloaded model RMSE: 52.6204
Reloaded model R2  : 0.4871


## 8. Promote the Model to Production

Add a description and transition the registered version to the **Production** stage. On DagsHub, the Models tab will now show this version tagged as Production.

In [16]:
client.update_model_version(
    name=model_name,
    version=latest_version.version,
    description="""
    Champion model for Diabetes disease-progression regression.

    Tested successfully before production deployment.

    Dataset: sklearn Diabetes
    Selection Metric: R2 Score
    """
)

<ModelVersion: aliases=[], creation_timestamp=1786027660760, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description=('\n'
 '    Champion model for Diabetes disease-progression regression.\n'
 '\n'
 '    Tested successfully before production deployment.\n'
 '\n'
 '    Dataset: sklearn Diabetes\n'
 '    Selection Metric: R2 Score\n'
 '    '), last_updated_timestamp=1786027689483, metrics=None, model_id=None, name='Diabetes_Best_Model', params=None, run_id='fe3bd040d94d46d290288e4cdfb6e3c6', run_link='', source='models:/m-1b785d4a2c1c4828a132952748bfebdc', status='READY', status_message=None, tags={}, user_id='', version='1', workspace='default'>

In [17]:
client.transition_model_version_stage(
    name=model_name,
    version=latest_version.version,
    stage="Production"
)

print("Model promoted to Production")

Model promoted to Production


In [18]:
production_model = mlflow.pyfunc.load_model(
    f"models:/{model_name}/Production"
)

prod_predictions = production_model.predict(X_test)
print("Production model first 10 predictions:", np.round(prod_predictions[:10], 2))

Production model first 10 predictions: [160.3  176.97 142.26 248.5  114.01 119.71 249.82 212.23 134.01 171.91]


In [19]:
production_versions = client.get_latest_versions(
    model_name,
    stages=["Production"]
)

for m in production_versions:
    print("Version :", m.version)
    print("Run ID  :", m.run_id)
    print("Stage   :", m.current_stage)

Version : 1
Run ID  : fe3bd040d94d46d290288e4cdfb6e3c6
Stage   : Production


## Done

DagsHub repo now contains:
- **Experiments tab** — one run per model with RMSE / MAE / R2
- **Models tab** — `Diabetes_Best_Model` registered, with a version promoted to **Production**

- Experiments page URL : https://dagshub.com/Nithilan77/DevOps/experiments
- Registered model page URL : https://dagshub.com/Nithilan77/DevOps/models